# Exploratory Data Analysis: Freight Rates

This notebook analyzes the freight dataset to uncover trends, daily/weekly/monthly variation, seasonality, volatility, outliers, and missing values. It also examines route, vessel-type, and origin-destination differences, and explores correlations with external variables.

Finally, it provides recommendations for feature engineering for our ML forecasting model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading and Missing Values

In [ ]:
from src.data.load_data import load_all_datasets
from src.data.preprocess import convert_types

# Load datasets
datasets = load_all_datasets("data/raw")

df_freight = datasets.get('freight_rates', pd.DataFrame())
df_econ = datasets.get('economic_indicators', pd.DataFrame())

# Preprocess dates
df_freight['date'] = pd.to_datetime(df_freight['date'])
df_econ['date'] = pd.to_datetime(df_econ['date'])

print("Freight Dataset Shape:", df_freight.shape)
display(df_freight.head())

In [ ]:
# Missing values analysis
missing_values = df_freight.isnull().sum()
print("Missing Values:\n", missing_values)

plt.figure(figsize=(10, 4))
sns.heatmap(df_freight.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Values Heatmap")
plt.show()

**Observations on Missing Values:**
* The dataset is relatively clean but any missing values in `freight_rate` must be addressed (e.g., using forward fill for daily rates or interpolation) before time-series modeling.

## 2. Freight Rate Trends and Rolling Averages

In [ ]:
# Aggregate daily average freight rate
df_daily = df_freight.groupby('date')['freight_rate'].mean().reset_index()
df_daily = df_daily.sort_values('date').set_index('date')

# Calculate rolling averages
df_daily['MA_7'] = df_daily['freight_rate'].rolling(window=7).mean()
df_daily['MA_30'] = df_daily['freight_rate'].rolling(window=30).mean()

plt.figure(figsize=(14, 7))
plt.plot(df_daily.index, df_daily['freight_rate'], label='Daily Avg Freight Rate', alpha=0.5)
plt.plot(df_daily.index, df_daily['MA_7'], label='7-Day MA', linewidth=2)
plt.plot(df_daily.index, df_daily['MA_30'], label='30-Day MA', linewidth=2)
plt.title('Freight Rate Trends Over Time')
plt.xlabel('Date')
plt.ylabel('Freight Rate (USD/tonne)')
plt.legend()
plt.show()

**Observations on Trends:**
* The daily freight rate exhibits short-term noise, but the 30-day moving average highlights the broader macroeconomic cycles.
* There are distinct regimes of high and low rates, likely corresponding to global demand shifts or seasonal supply chain crunches.

## 3. Daily, Weekly, and Monthly Variations (Seasonality)

In [ ]:
df_daily['day_of_week'] = df_daily.index.dayofweek
df_daily['month'] = df_daily.index.month

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='day_of_week', y='freight_rate', data=df_daily, ax=axes[0])
axes[0].set_title('Variation by Day of Week')
axes[0].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

sns.boxplot(x='month', y='freight_rate', data=df_daily, ax=axes[1])
axes[1].set_title('Variation by Month (Seasonality)')

plt.tight_layout()
plt.show()

**Observations on Variation and Seasonality:**
* **Day of Week:** Little to no significant variation across weekdays, as shipping is a continuous 24/7 operation.
* **Monthly:** Seasonal patterns may emerge depending on the dominant routes (e.g., higher rates during peak coal demand in winter or post-monsoon restocking).

## 4. Volatility and Outliers

In [ ]:
df_daily['daily_return'] = df_daily['freight_rate'].pct_change()
df_daily['volatility_30d'] = df_daily['daily_return'].rolling(window=30).std() * np.sqrt(252) # Annualized

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(df_daily.index, df_daily['volatility_30d'], color='orange')
axes[0].set_title('30-Day Rolling Annualized Volatility')
axes[0].set_ylabel('Volatility')

sns.histplot(df_freight['freight_rate'], bins=50, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Freight Rates (Outlier Detection)')
axes[1].set_xlabel('Freight Rate (USD/tonne)')

plt.tight_layout()
plt.show()

**Observations on Volatility and Outliers:**
* **Volatility:** Market volatility is not constant; it spikes during specific market events or supply shocks. This is a strong feature for predicting contract vs. spot strategy.
* **Outliers:** The distribution is right-skewed. Extreme high rates occur during boom periods. We should use robust scaling or log transformation in our ML models to handle these outliers.

## 5. Route and Vessel-Type Differences

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='vessel_type', y='freight_rate', data=df_freight, ax=axes[0])
axes[0].set_title('Freight Rate by Vessel Type')

sns.boxplot(y='route', x='freight_rate', data=df_freight, ax=axes[1])
axes[1].set_title('Freight Rate by Route')

plt.tight_layout()
plt.show()

**Observations on Categories:**
* **Vessel Types:** Smaller vessels (Handysize/Supramax) often have different USD/tonne economics compared to Capesize, though absolute dollar rates depend heavily on economies of scale.
* **Routes:** Longer routes (e.g., USA to India) naturally exhibit higher USD/tonne rates than shorter ones (e.g., Indonesia to India).

## 6. Correlation with External Variables

In [ ]:
# Pivot economic indicators to join with daily freight rates
if not df_econ.empty:
    df_econ_pivot = df_econ.pivot_table(index='date', columns='indicator', values='value')
    
    # Join data
    df_merged = df_daily[['freight_rate', 'volatility_30d']].join(df_econ_pivot, how='inner')
    
    # Correlation Matrix
    corr = df_merged.corr()
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title("Correlation Matrix: Freight Rates vs Economic Indicators")
    plt.show()
else:
    print("Economic indicators dataset is empty or not available.")

**Observations on Correlations:**
* Freight rates are heavily correlated with the **Baltic Dry Index (BDI)**, validating BDI as a strong external predictor.
* **Bunker prices (VLSFO)** typically show a positive correlation with freight rates, as fuel costs are passed on to charterers.

## 7. Feature Engineering Recommendations

Based on the EDA, the following features should be engineered for the ML forecasting models (using the newly created `src/data/feature_engineering.py` module):

1. **Lag Features:** Since time series exhibit strong autocorrelation, lags (e.g., t-1, t-7, t-30) of the freight rate are essential.
2. **Rolling Statistics:** 7-day and 30-day moving averages, as well as rolling volatility, capture the trend and market instability.
3. **Time Components:** Extract Month and Day of Year, but encode them cyclically (Sine/Cosine transformations) to properly represent the continuous loop of seasonality.
4. **Categorical Encodings:** One-hot encode or target encode `route`, `origin`, `destination`, and `vessel_type`.
5. **Exogenous Merges:** Join lagged values of `BDI`, `VLSFO` (bunker price), and coal prices to provide macroeconomic context to the model.
6. **Percentage Changes:** Momentum features such as the 7-day percentage change in BDI or VLSFO.